In [1]:
import mne 
import torch
import numpy as np
import os 
from sklearn.model_selection import train_test_split

In [2]:
subject_ids = range(1,110)
all_x,all_y = [],[]

In [3]:
base_path = "E:/EEG_Dataset/eeg-motor-movementimagery-dataset-1.0.0"

In [4]:
for subject_id in subject_ids:
    subj = f"S{subject_id:03d}"
    
    for run in ["R04", "R08", "R12"]:
        edf_path = os.path.join(base_path, subj, f"{subj}{run}.edf")
        
        

        raw = mne.io.read_raw_edf(edf_path, preload=True)
        raw.rename_channels(lambda s: s.replace('.','').upper())
        raw.pick_channels([
            "C3", "C4", "CZ", "CP1", "CP2", "FC1", "FC2", "FZ"
        ])
        
        raw.filter(1., 30., fir_design='firwin')

        events, _ = mne.events_from_annotations(raw)

        event_id = {'T1': 1, 'T2': 2}

        epochs = mne.Epochs(raw, events, event_id=event_id,
                                tmin=-0.5, tmax=3.5,
                                baseline=None, preload=True)


        x = epochs.get_data()
        y = epochs.events[:, -1]
        
        all_x.append(x)
        all_y.append(y)
        print(x.shape)

Extracting EDF parameters from E:\EEG_Dataset\eeg-motor-movementimagery-dataset-1.0.0\S001\S001R04.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 529 samples (3.306 s)

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Not setting metadata
23 matching event

C:\Users\kgtk0\AppData\Local\Temp\ipykernel_1724\4137465547.py:9: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_path, preload=True)
C:\Users\kgtk0\AppData\Local\Temp\ipykernel_1724\4137465547.py:9: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_path, preload=True)
C:\Users\kgtk0\AppData\Local\Temp\ipykernel_1724\4137465547.py:9: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = mne.io.read_raw_edf(edf_path, preload=True)


Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]
Not setting metadata
18 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 18 events and 513 original time points ...
1 bad epochs dropped
(17, 8, 513)
Extracting EDF parameters from E:\EEG_Dataset\eeg-motor-movementimagery-dataset-1.0.0\S101\S101R04.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz 

In [5]:
all_x_fixed = []
all_y_fixed = []
for x, y in zip(all_x, all_y):
    if x.shape[2] == 641:
        all_x_fixed.append(x)
        all_y_fixed.append(y)

x_all = np.concatenate(all_x_fixed, axis=0)
y_all = np.concatenate(all_y_fixed, axis=0)


print(x_all.shape)  # 例：(試行数, 8チャネル, 時間点数)
print(y_all.shape)  # (試行数,)

(6855, 8, 641)
(6855,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    x_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

In [7]:
# 正規化
X_train = (X_train - X_train.mean(axis=2, keepdims=True)) / X_train.std(axis=2, keepdims=True)
X_test  = (X_test  - X_test.mean(axis=2, keepdims=True))  / X_test.std(axis=2, keepdims=True)

In [8]:
from torch.utils.data import Dataset

class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, C, T)
        self.y = torch.tensor(y-1, dtype=torch.long)     # (N,)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [9]:
from torch.utils.data import DataLoader

train_dataset = EEGDataset(X_train, y_train)
test_dataset = EEGDataset(X_test, y_test)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [10]:
import torch.nn as nn
import torch.nn.functional as F

class EEGNet(nn.Module):
    def __init__(self, num_classes=2, channels=16, samples=641):
        super(EEGNet, self).__init__()

        # 第1層: Temporal Convolution
        self.firstconv = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=(1, 64), padding=(0, 32), bias=False),
            nn.BatchNorm2d(8)
        )

        # 第2層: Depthwise Convolution (Spatial filter)
        self.depthwiseConv = nn.Sequential(
            nn.Conv2d(8, 16, kernel_size=(channels, 1), groups=8, bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(p=0.25)
        )

        # 第3層: Separable Convolution
        self.separableConv = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=(1, 16), padding=(0, 8), bias=False),
            nn.BatchNorm2d(16),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 8)),
            nn.Dropout(p=0.25)
        )

        # 全結合層
        self.classify = nn.Linear(self._get_flattened_size(channels, samples), num_classes)

    def _get_flattened_size(self, channels, samples):
        x = torch.zeros(1, 1, channels, samples)
        x = self.firstconv(x)
        x = self.depthwiseConv(x)
        x = self.separableConv(x)
        return x.numel()

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, C, T)
        x = self.firstconv(x)
        x = self.depthwiseConv(x)
        x = self.separableConv(x)
        x = x.view(x.size(0), -1)
        return self.classify(x)

In [11]:
device = torch.device("cuda")  

model = EEGNet(num_classes=2, channels=8, samples=641).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)


In [12]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y).sum().item()
        total += y.size(0)

    acc = correct / total
    return running_loss / len(loader), acc

def evaluate(model, loader, criterion, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == y).sum().item()
            total += y.size(0)

    return correct / total

In [13]:
epochs = 50  # 時間と相談して増減してOK

for epoch in range(epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    test_acc = evaluate(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

Epoch 1/50 - Train Loss: 0.6380, Train Acc: 0.6329, Test Acc: 0.6966
Epoch 2/50 - Train Loss: 0.5697, Train Acc: 0.7113, Test Acc: 0.7316
Epoch 3/50 - Train Loss: 0.5371, Train Acc: 0.7316, Test Acc: 0.7403
Epoch 4/50 - Train Loss: 0.5049, Train Acc: 0.7560, Test Acc: 0.7702
Epoch 5/50 - Train Loss: 0.4907, Train Acc: 0.7591, Test Acc: 0.7812
Epoch 6/50 - Train Loss: 0.4761, Train Acc: 0.7730, Test Acc: 0.7921
Epoch 7/50 - Train Loss: 0.4678, Train Acc: 0.7812, Test Acc: 0.7892
Epoch 8/50 - Train Loss: 0.4573, Train Acc: 0.7832, Test Acc: 0.7950
Epoch 9/50 - Train Loss: 0.4525, Train Acc: 0.7867, Test Acc: 0.7929
Epoch 10/50 - Train Loss: 0.4502, Train Acc: 0.7868, Test Acc: 0.7936
Epoch 11/50 - Train Loss: 0.4416, Train Acc: 0.7950, Test Acc: 0.8031
Epoch 12/50 - Train Loss: 0.4388, Train Acc: 0.7972, Test Acc: 0.8045
Epoch 13/50 - Train Loss: 0.4344, Train Acc: 0.7985, Test Acc: 0.8096
Epoch 14/50 - Train Loss: 0.4390, Train Acc: 0.7961, Test Acc: 0.8053
Epoch 15/50 - Train Loss: 0.4

In [14]:
# モデルの重みだけ保存
#torch.save(model.state_dict(), 'eegnet_model.pt')
